In [ ]:
# =============================================================================
# Bus Fare Cap Policy and School Attendance
# Analysis script: data preparation, DiD model, t-tests, and visualisations
#
# Data sources:
#   - DfE school attendance data (termly, LA-level)
#   - DfT bus usage data (annual, LA-level)
#   - Census 2021 car ownership data
#   - Rural/Urban classification (2021)
#   - LA to Combined Authority mapping (manual)
#   - London borough boundary GeoJSON
#
# Structure:
#   1.  Imports
#   2.  Paths  <-- UPDATE THESE for your machine
#   3.  Attendance data preparation
#   4.  Bus usage data preparation
#   5.  Car ownership data preparation
#   6.  Rural/Urban classification preparation
#   7.  Merge all datasets into analysis-ready file
#   8.  Difference-in-Differences (DiD) model
#   9.  Paired t-tests (Home Counties LAs)
#   10. Exploratory FSM t-test (national)
#   11. Visualisations: London choropleth map
#   12. Visualisations: London FSM trend line chart
#   13. Visualisations: London borough bar chart (interactive)
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
from pathlib import Path
from scipy import stats
from scipy.stats import ttest_ind

from bokeh.io import output_notebook, output_file, show, save
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource, GeoJSONDataSource,
    LinearColorMapper, ColorBar,
    HoverTool, Select, CustomJS, Legend
)
from bokeh.layouts import column, row
from bokeh.palettes import Category10

output_notebook()


# =============================================================================
# 2. PATHS
# =============================================================================

# raw_dir = Path("C:/Users/Max/Documents/bus_fares_attendance/data/raw")
# derived_dir = Path("C:/Users/Max/Documents/bus_fares_attendance/data/derived")
# notebooks_dir = Path("C:/Users/Max/Documents/bus_fares_attendance/notebooks")

# Uncomment and set your own paths:
# raw_dir = Path("/path/to/your/data/raw")
# derived_dir = Path("path/to/your/data/derived")
# notebooks_dir = Path("/path/to/your/notebooks")

# Ensure derived and notebooks folders exist
derived_dir.mkdir(parents=True, exist_ok=True)
notebooks_dir.mkdir(parents=True, exist_ok=True)

# Individual file paths (raw inputs)
# attendance_dir = raw_dir / "attendance"         # folder of per-year attendance CSVs
# bus_path       = raw_dir / "bus01.ods"          # DfT bus usage table BUS01f
# car_path       = raw_dir / "TS045-2021-4.csv"   # Census 2021 car ownership
# rural_path     = raw_dir / "Rural_Urban_Classification_(2021)_of_Local_Authority_Districts_(2021)_in_EW.csv"
# la_to_ca_path  = raw_dir / "la_to_ca.csv"       # manual LA → CA mapping
# boundary_path  = derived_dir / "Local_Authority_Districts_December_2023_Boundaries_UK_BFE_1230766033994939568.geojson"


# =============================================================================
# 3. ATTENDANCE DATA PREPARATION
# =============================================================================

# Columns to retain from each annual attendance file
cols_keep = [
    "time_period", "time_identifier", "geographic_level",
    "country_code", "country_name",
    "region_code", "region_name",
    "new_la_code", "la_name",
    "school_type", "disadvantage_code",
    "overall_absence_perc", "unauthorised_absence_perc", "pa_perc"
]

# Some years use different column names — harmonise before stacking
rename_map = {
    "education_phase": "school_type",
    "free_school_meals_indicator": "disadvantage_code"
}

# Load, harmonise, and stack all annual files
dfs = []
for file in list(attendance_dir.rglob("*.csv")):
    df = pd.read_csv(file)
    df = df.rename(columns=rename_map)
    df = df[cols_keep]
    dfs.append(df)

attendance_all = pd.concat(dfs, ignore_index=True)

# Keep only Local Authority rows (drop national/regional aggregates)
attendance_all = attendance_all[
    attendance_all["geographic_level"] == "Local authority"
].copy()

# Assign calendar year: Autumn term belongs to the start year; Spring/Summer to year+1
def term_to_year(row):
    start_year = int(str(row["time_period"])[:4])
    return start_year if row["time_identifier"] == "Autumn term" else start_year + 1

attendance_all["year"] = attendance_all.apply(term_to_year, axis=1)

# Load LA → CA mapping and merge onto attendance
ca_map = pd.read_csv(la_to_ca_path)

attendance_all = attendance_all.merge(
    ca_map[["la_code", "ca_code", "ca"]],
    left_on="new_la_code",
    right_on="la_code",
    how="left"
)

# Flag London rows; collapse all 33 London LAs into a single "London" analysis unit
attendance_all["is_london"] = attendance_all["region_name"] == "London"

attendance_all["analysis_unit_name"] = attendance_all.apply(
    lambda row: "London" if row["is_london"]
    else row["ca"] if pd.notna(row["ca"])
    else row["la_name"],
    axis=1
)

attendance_all["analysis_unit_code"] = attendance_all.apply(
    lambda row: "E12000007" if row["is_london"]
    else row["ca_code"] if pd.notna(row["ca_code"])
    else row["new_la_code"],
    axis=1
)

# Ensure absence columns are numeric
for col in ["overall_absence_perc", "unauthorised_absence_perc", "pa_perc"]:
    attendance_all[col] = pd.to_numeric(attendance_all[col], errors="coerce")

# Aggregate to analysis unit / year / term / school type / disadvantage group
attendance_agg = (
    attendance_all
    .groupby([
        "analysis_unit_code", "analysis_unit_name",
        "year", "time_identifier",
        "school_type", "disadvantage_code"
    ], as_index=False)
    .agg({
        "overall_absence_perc": "mean",
        "unauthorised_absence_perc": "mean",
        "pa_perc": "mean"
    })
)

# Save cleaned attendance dataset
attendance_agg.to_csv(derived_dir / "attendance_clean.csv", index=False)


# =============================================================================
# 4. BUS USAGE DATA PREPARATION
# =============================================================================

# Load DfT bus usage table (skip top 8 descriptive header rows)
bus_raw = pd.read_excel(bus_path, sheet_name="BUS01f", engine="odf", skiprows=8)

# Retain geography columns and the relevant years only
bus_cols = [
    bus_raw.columns[0],   # LA / Region code
    bus_raw.columns[1],   # LA / Region name
    "2022", "2023", "2024", "2025"
]
bus = bus_raw[bus_cols].copy()

# Rename geography columns for clarity before any further operations
bus = bus.rename(columns={
    bus.columns[0]: "analysis_unit_code",
    bus.columns[1]: "analysis_unit_name"
})

# Drop regional aggregates (E12xxx codes), but keep London; drop England total ([z])
bus = bus[
    ~(
        (bus["analysis_unit_code"].str.startswith("E12") &
         (bus["analysis_unit_name"] != "London"))
        |
        (bus["analysis_unit_code"] == "[z]")
    )
].copy()

# Reshape from wide (one column per year) to long (one row per area/year)
bus_long = bus.melt(
    id_vars=["analysis_unit_code", "analysis_unit_name"],
    value_vars=["2022", "2023", "2024", "2025"],
    var_name="year",
    value_name="bus_usage"
)

bus_long["year"] = bus_long["year"].astype(int)
bus_long["bus_usage"] = pd.to_numeric(bus_long["bus_usage"], errors="coerce")

# Save cleaned bus dataset
bus_long.to_csv(derived_dir / "bus_clean_long.csv", index=False)

# Optional sanity checks (uncomment if needed):
# print(bus_long.head())
# print(bus_long.isna().sum())
# print(bus_long["analysis_unit_code"].nunique())


# =============================================================================
# 5. CAR OWNERSHIP DATA PREPARATION
# =============================================================================

# Load Census 2021 car ownership data
car_df = pd.read_csv(car_path)

# Load LA → CA mapping (reuse la_to_ca_path)
la_to_ca = pd.read_csv(la_to_ca_path)

# Merge CA information onto car data
car_with_ca = car_df.merge(
    la_to_ca[["la_code", "ca_code", "ca"]],
    left_on="Lower tier local authorities Code",
    right_on="la_code",
    how="left"
)

# Use CA if present, otherwise retain original LA code/name
car_with_ca["analysis_unit_code"] = car_with_ca["ca_code"].combine_first(
    car_with_ca["Lower tier local authorities Code"]
)
car_with_ca["analysis_unit_name"] = car_with_ca["ca"].combine_first(
    car_with_ca["Lower tier local authorities"]
)

# Aggregate to analysis unit
car_final = (
    car_with_ca
    .groupby(["analysis_unit_code", "analysis_unit_name"], as_index=False)["Observation"]
    .mean()
    .rename(columns={"Observation": "car_ownership"})
)

# Add a year column for alignment with attendance and bus datasets
# (car ownership data is from 2021 only; cross-joined to all years at merge stage)
car_final["year"] = 2021

# Keep only units present in the attendance data (removes non-English LAs)
attendance_clean = pd.read_csv(derived_dir / "attendance_clean.csv")
attendance_units = attendance_clean["analysis_unit_code"].unique()
car_final = car_final[car_final["analysis_unit_code"].isin(attendance_units)].copy()

# Save cleaned car ownership dataset
car_final.to_csv(derived_dir / "car_ownership_clean.csv", index=False)


# =============================================================================
# 6. RURAL/URBAN CLASSIFICATION PREPARATION
# =============================================================================

# Load rural/urban classification
rural_df = pd.read_csv(rural_path)

# Merge LA → CA mapping
rural_df = rural_df.merge(
    la_to_ca[["la_code", "ca_code", "ca", "region"]],
    left_on="LAD21CD",
    right_on="la_code",
    how="left"
)

# Create analysis unit columns (CA if present, else LA)
rural_df["analysis_unit_code"] = rural_df["ca_code"].combine_first(rural_df["LAD21CD"])
rural_df["analysis_unit_name"] = rural_df["ca"].combine_first(rural_df["LAD21NM"])

# Collapse all London LAs into a single London unit
london_las = la_to_ca.loc[la_to_ca["region"] == "London", "la_code"].tolist()
rural_df.loc[rural_df["LAD21CD"].isin(london_las), "analysis_unit_code"] = "E12000007"
rural_df.loc[rural_df["LAD21CD"].isin(london_las), "analysis_unit_name"] = "London"

# Aggregate by analysis unit, taking the modal rural/urban classification
rural_final = (
    rural_df
    .groupby(["analysis_unit_code", "analysis_unit_name"], as_index=False)
    .agg({
        "Urban_rural_flag": lambda x: x.mode()[0],
        "RUC21CD": lambda x: x.mode()[0],
        "RUC21NM": lambda x: x.mode()[0]
    })
)

# Keep only units present in the attendance data
rural_final = rural_final[
    rural_final["analysis_unit_code"].isin(attendance_clean["analysis_unit_code"])
]

# Add year column (2021 census; cross-joined to all years at merge stage)
rural_final["year"] = 2021

# Save cleaned rural/urban dataset
rural_final.to_csv(derived_dir / "rural_urban_clean.csv", index=False)


# =============================================================================
# 7. MERGE ALL DATASETS INTO ANALYSIS-READY FILE
# =============================================================================

# Reload cleaned datasets
attendance_clean = pd.read_csv(derived_dir / "attendance_clean.csv")
bus_clean_long   = pd.read_csv(derived_dir / "bus_clean_long.csv")
car_final        = pd.read_csv(derived_dir / "car_ownership_clean.csv")
rural_final      = pd.read_csv(derived_dir / "rural_urban_clean.csv")

# Drop duplicate name columns from proxy datasets (name is carried from attendance)
for df in [bus_clean_long, car_final, rural_final]:
    df.drop(columns=["analysis_unit_name"], errors="ignore", inplace=True)

# Car ownership and rural/urban are single-year snapshots (2021);
# cross-join with all attendance years so they can be merged on (unit, year)
years = attendance_clean["year"].unique()

car_expanded = (
    car_final[["analysis_unit_code", "car_ownership"]]
    .merge(pd.DataFrame({"year": years}), how="cross")
)

rural_expanded = (
    rural_final[["analysis_unit_code", "Urban_rural_flag"]]
    .merge(pd.DataFrame({"year": years}), how="cross")
)

# Merge all datasets onto attendance (left joins to preserve all attendance rows)
merged_df = (
    attendance_clean
    .merge(bus_clean_long,  on=["analysis_unit_code", "year"], how="left", validate="m:1")
    .merge(car_expanded,    on=["analysis_unit_code", "year"], how="left", validate="m:1")
    .merge(rural_expanded,  on=["analysis_unit_code", "year"], how="left", validate="m:1")
)

# Save final analysis-ready dataset
merged_df.to_csv(derived_dir / "analysis_ready.csv", index=False)

# Optional sanity check (uncomment if needed):
# print(f"Merged dataset rows: {len(merged_df)}")
# print("Columns:", merged_df.columns.tolist())
# print(merged_df.head())


# =============================================================================
# 8. DIFFERENCE-IN-DIFFERENCES (DiD) MODEL
#
# Tests whether areas with higher bus usage saw greater changes in school
# attendance during the bus fare cap period (2023-24) and after its removal
# (2025), compared to lower-usage areas.
#
# NOTE: The DiD approach was originally designed with London as a control group,
# but this was abandoned due to insufficient variance in the London data.
# This version uses bus usage as a continuous exposure variable instead.
# =============================================================================

df = pd.read_csv(derived_dir / "analysis_ready.csv")

dep_var = "overall_absence_perc"

# Define policy periods
df["pre_cap"]    = (df["year"] < 2023).astype(int)
df["cap_period"] = ((df["year"] >= 2023) & (df["year"] <= 2024)).astype(int)
df["post_cap"]   = (df["year"] >= 2025).astype(int)

# Binary exposure variable: above-median bus usage = high exposure to fare cap
df["high_exposure"] = (df["bus_usage"] >= df["bus_usage"].median()).astype(int)

# DiD interaction terms: cap/post period × exposure
df["cap_x_exposure"]  = df["cap_period"] * df["high_exposure"]
df["post_x_exposure"] = df["post_cap"]   * df["high_exposure"]

# Drop rows with missing values in key regression columns
reg_cols = [
    dep_var, "cap_x_exposure", "post_x_exposure",
    "analysis_unit_code", "year", "school_type", "disadvantage_code"
]
df_reg = df.dropna(subset=reg_cols).copy()

print(f"Rows before dropna: {len(df)}, after dropna: {len(df_reg)}")

# OLS regression with area and year fixed effects, clustered standard errors by area
formula = (
    f"{dep_var} ~ cap_x_exposure + post_x_exposure "
    "+ C(analysis_unit_code) + C(year) + C(school_type) + C(disadvantage_code)"
)

model = smf.ols(formula=formula, data=df_reg).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_reg["analysis_unit_code"]}
)

print(model.summary())


# =============================================================================
# 9. PAIRED T-TESTS (HOME COUNTIES LAs)
#
# After the DiD approach failed, a paired t-test design was used instead.
# The same ~60 Home Counties LAs are compared before and after the fare cap
# (Test 1) and before and after its removal (Test 2).
# Results were inconclusive: the post-Covid attendance recovery trend was
# likely a confound, not independent of the policy intervention.
#
# Three versions below reflect iterative refinement of the LA sample.
# =============================================================================

# --- Version 1: 10 target LAs ---
df_home = pd.read_csv(derived_dir / "analysis_ready.csv")
df_home["post"] = (df_home["year"] >= 2023).astype(int)

target_LAs_v1 = [
    "Hertfordshire", "Buckinghamshire", "Surrey", "Kent", "Berkshire",
    "Essex", "Sussex", "Hampshire", "Luton", "Oxfordshire"
]
df_subset_v1 = df_home[df_home["analysis_unit_name"].isin(target_LAs_v1)]

formula_fe = (
    "overall_absence_perc ~ post "
    "+ C(analysis_unit_name) + C(year) + C(school_type) + C(disadvantage_code)"
)

model_v1 = smf.ols(formula=formula_fe, data=df_subset_v1).fit(cov_type="HC3")
print("--- Version 1: 10 LAs ---")
print(model_v1.summary())

# --- Version 2: Reduced to 6 LAs (trimmed sample) ---
target_LAs_v2 = [
    "Hertfordshire", "Buckinghamshire", "Surrey", "Kent", "Berkshire", "Essex"
]
df_subset_v2 = df_home[df_home["analysis_unit_name"].isin(target_LAs_v2)]

model_v2 = smf.ols(formula=formula_fe, data=df_subset_v2).fit(cov_type="HC3")
print("--- Version 2: 6 LAs ---")
print(model_v2.summary())

# --- Version 3: Cap-on vs post-cap year only (2023-2025) ---
df_test2 = df_home[df_home["year"].isin([2023, 2024, 2025])].copy()
df_test2["post_cap"] = (df_test2["year"] == 2025).astype(int)

formula_postcap = (
    "overall_absence_perc ~ post_cap "
    "+ C(analysis_unit_name) + C(year) + C(school_type) + C(disadvantage_code)"
)

model_2025 = smf.ols(formula=formula_postcap, data=df_test2).fit(cov_type="HC3")
print("--- Version 3: Cap-on vs post-cap (2023-2025) ---")
print(model_2025.summary())


# =============================================================================
# 10. EXPLORATORY FSM T-TEST (NATIONAL)
#
# Quick check: are FSM-eligible pupils consistently more absent than
# non-FSM pupils across terms? Treated as a rough descriptive comparison
# only — does not control for LA, bus exposure, or policy period.
# =============================================================================

# Version A: using manually specified term-level values
fsm_values     = [14.1, 14.2, 16, 13.2, 14.4, 15.8, 12.8, 14.1, 15]
non_fsm_values = [7.8,  7.2,  8,  6.2,  7.0,  7.8,  6.0,  6.6,  7.2]

t_stat, p_val = ttest_ind(fsm_values, non_fsm_values)
print(f"[Exploratory t-test A] T-statistic: {t_stat:.2f}, P-value: {p_val:.4f}")

# Version B: using the full attendance dataset
attendance_national = pd.read_csv(derived_dir / "attendance_all.csv")
attendance_national = attendance_national[
    attendance_national["disadvantage_code"].isin(["FSM eligible", "FSM Not eligible"])
].copy()
attendance_national["overall_absence_perc"] = pd.to_numeric(
    attendance_national["overall_absence_perc"], errors="coerce"
)
attendance_national = attendance_national.dropna(subset=["overall_absence_perc"])

fsm_nat     = attendance_national.loc[attendance_national["disadvantage_code"] == "FSM eligible",     "overall_absence_perc"]
non_fsm_nat = attendance_national.loc[attendance_national["disadvantage_code"] == "FSM Not eligible", "overall_absence_perc"]

t_stat_b, p_val_b = stats.ttest_ind(fsm_nat, non_fsm_nat, equal_var=False, nan_policy="omit")

print("\n[Exploratory t-test B] FSM vs non-FSM absence (national)")
print(f"Mean absence (FSM eligible):     {fsm_nat.mean():.2f}")
print(f"Mean absence (FSM not eligible): {non_fsm_nat.mean():.2f}")
print(f"Difference (pp):                 {(fsm_nat.mean() - non_fsm_nat.mean()):.2f}")
print(f"T-statistic:                     {t_stat_b:.2f}")
print(f"P-value:                         {p_val_b:.4g}")


# =============================================================================
# 11. LONDON CHOROPLETH MAP
#
# Interactive HTML map showing average overall absence by London borough.
# Saved as a standalone HTML file for embedding in the policy paper.
# =============================================================================

# Load boundary data and attendance
gdf = gpd.read_file(boundary_path)
attendance_map = pd.read_csv(derived_dir / "attendance_all.csv")

# Keep LA-level rows only
attendance_map = attendance_map[
    attendance_map["new_la_code"].notna() &
    (attendance_map["new_la_code"].astype(str).str.strip() != "")
].copy()

# Filter to 33 London boroughs (E09000001 to E09000033)
london_codes = {f"E090000{str(i).zfill(2)}" for i in range(1, 34)}
gdf_lon = gdf[gdf["LAD23CD"].isin(london_codes)].copy()
att_lon = attendance_map[attendance_map["new_la_code"].isin(london_codes)].copy()

# Compute mean absence per borough
METRIC = "overall_absence_perc"
att_lon[METRIC] = pd.to_numeric(att_lon[METRIC], errors="coerce")
att_lon = att_lon.dropna(subset=[METRIC])

la_metric_lon = (
    att_lon.groupby("new_la_code", as_index=False)[METRIC]
    .mean()
    .rename(columns={METRIC: "metric_value"})
)

# Merge metric onto boundary polygons
gdf_lon_merged = gdf_lon.merge(
    la_metric_lon, left_on="LAD23CD", right_on="new_la_code", how="left"
)

# Retain only needed fields and simplify geometry to reduce file size
gdf_lon_merged = gdf_lon_merged[["LAD23CD", "LAD23NM", "metric_value", "geometry"]].copy()
gdf_tmp = gdf_lon_merged.to_crs(epsg=3857)
gdf_tmp["geometry"] = gdf_tmp["geometry"].simplify(tolerance=50, preserve_topology=True)
gdf_lon_merged = gdf_tmp.to_crs(epsg=4326)

print(f"London polygons in boundary file: {len(gdf_lon_merged)}")
print(f"London polygons with data:        {gdf_lon_merged['metric_value'].notna().sum()}")

# Build Bokeh choropleth
geo_source = GeoJSONDataSource(geojson=gdf_lon_merged.to_json())
vals = gdf_lon_merged["metric_value"].dropna()
color_mapper = LinearColorMapper(
    palette="Viridis256", low=float(vals.min()), high=float(vals.max())
)

p_map = figure(
    title="London: average absence by local authority",
    tools="pan,wheel_zoom,reset",
    toolbar_location="above",
    match_aspect=True
)

p_map.patches(
    "xs", "ys",
    source=geo_source,
    fill_color={"field": "metric_value", "transform": color_mapper},
    line_color="white", line_width=0.7
)

p_map.add_tools(HoverTool(tooltips=[
    ("LA", "@LAD23NM"),
    ("Code", "@LAD23CD"),
    ("Absence (%)", "@metric_value{0.00}")
]))

color_bar = ColorBar(color_mapper=color_mapper, label_standoff=12, title="Absence (%)")
p_map.add_layout(color_bar, "right")

# Save as standalone HTML
output_file(
    str(notebooks_dir / "london_heatmap_absence.html"),
    title="London heatmap: absence by local authority"
)
save(p_map)
print("Saved:", notebooks_dir / "london_heatmap_absence.html")


# =============================================================================
# 12. LONDON FSM TREND LINE CHART
#
# Interactive line chart showing absence rates over time by FSM status,
# with a school type filter. Saved as a standalone HTML file.
# =============================================================================

attendance_line = pd.read_csv(derived_dir / "attendance_all.csv")

# Filter to London LA-level rows
attendance_line = attendance_line[
    (attendance_line["region_name"] == "London") &
    (attendance_line["geographic_level"] == "Local authority") &
    attendance_line["new_la_code"].notna() &
    (attendance_line["new_la_code"].astype(str).str.strip() != "")
].copy()

# Keep FSM groups only; clean fields
for col in ["school_type", "disadvantage_code", "time_identifier"]:
    attendance_line[col] = attendance_line[col].astype(str).str.strip()

attendance_line = attendance_line[
    attendance_line["disadvantage_code"].isin(["FSM eligible", "FSM Not eligible"])
].copy()

attendance_line["overall_absence_perc"] = pd.to_numeric(
    attendance_line["overall_absence_perc"], errors="coerce"
)
attendance_line = attendance_line.dropna(subset=["overall_absence_perc"])

# Build time index for correct x-axis ordering
term_order_map = {"Autumn term": 1, "Spring term": 2, "Summer term": 3}
attendance_line["term_order"] = attendance_line["time_identifier"].map(term_order_map)
attendance_line = attendance_line.dropna(subset=["term_order"])
attendance_line["time_period"] = attendance_line["time_period"].astype(int)
attendance_line["time_index"]  = attendance_line["time_period"] * 10 + attendance_line["term_order"]
attendance_line["term_label"]  = attendance_line["time_period"].astype(str) + " " + attendance_line["time_identifier"]

def build_trend(df):
    """Aggregate mean absence by term and FSM group; return trend df and ordered labels."""
    trend = (
        df.groupby(["time_index", "term_label", "disadvantage_code"], as_index=False)
          .agg(overall_absence=("overall_absence_perc", "mean"))
          .sort_values("time_index")
    )
    term_labels = (
        trend[["time_index", "term_label"]]
        .drop_duplicates()
        .sort_values("time_index")["term_label"]
        .tolist()
    )
    return trend, term_labels

# Pre-compute data for each school type option
options = ["All", "Primary", "Secondary", "Special"]
data_by_option = {}
for opt in options:
    df_opt = attendance_line if opt == "All" else attendance_line[attendance_line["school_type"] == opt].copy()
    data_by_option[opt] = build_trend(df_opt)

# Initialise plot with "All" school types
trend_df, term_labels_ordered = data_by_option["All"]
colors = Category10[10]

def subset_source(trend, group):
    return ColumnDataSource(trend[trend["disadvantage_code"] == group].copy())

source_fsm = subset_source(trend_df, "FSM eligible")
source_non = subset_source(trend_df, "FSM Not eligible")

p_line = figure(
    x_range=term_labels_ordered,
    height=420, width=950,
    title="London: overall absence rates by term and FSM status",
    x_axis_label="School term",
    y_axis_label="Average overall absence (%)",
    toolbar_location=None
)

p_line.line("term_label", "overall_absence", source=source_fsm, line_width=3, color=colors[0], legend_label="FSM eligible")
p_line.scatter("term_label", "overall_absence", source=source_fsm, size=7, marker="circle", color=colors[0])
p_line.line("term_label", "overall_absence", source=source_non, line_width=3, color=colors[1], legend_label="FSM Not eligible")
p_line.scatter("term_label", "overall_absence", source=source_non, size=7, marker="circle", color=colors[1])

p_line.legend.location = "center_right"
p_line.legend.orientation = "vertical"
p_line.legend.background_fill_alpha = 0.0
p_line.legend.border_line_alpha = 0.0
p_line.add_layout(p_line.legend[0], "right")
p_line.xaxis.major_label_orientation = 0.9

# Prepare JS data payload for interactive filter
js_data = {}
for opt in options:
    trend_opt, labels_opt = data_by_option[opt]
    df_fsm = trend_opt[trend_opt["disadvantage_code"] == "FSM eligible"]
    df_non = trend_opt[trend_opt["disadvantage_code"] == "FSM Not eligible"]
    js_data[opt] = {
        "labels": labels_opt,
        "fsm": {"term_label": df_fsm["term_label"].tolist(), "overall_absence": df_fsm["overall_absence"].tolist()},
        "non": {"term_label": df_non["term_label"].tolist(), "overall_absence": df_non["overall_absence"].tolist()}
    }

select_line = Select(title="School Type", value="All", options=options)

callback_line = CustomJS(
    args=dict(src_fsm=source_fsm, src_non=source_non, plot=p_line, data_map=js_data),
    code="""
    const opt = cb_obj.value;
    const d = data_map[opt];
    plot.x_range.factors = d.labels;
    src_fsm.data = { term_label: d.fsm.term_label, overall_absence: d.fsm.overall_absence };
    src_non.data = { term_label: d.non.term_label, overall_absence: d.non.overall_absence };
    src_fsm.change.emit();
    src_non.change.emit();
    plot.change.emit();
    """
)
select_line.js_on_change("value", callback_line)

layout_line = column(select_line, p_line)

# Save as standalone HTML (layout includes the dropdown filter)
output_file(
    str(notebooks_dir / "london_fsm_trend_linechart.html"),
    title="London: absence over time by FSM status"
)
save(layout_line)
print("Saved:", notebooks_dir / "london_fsm_trend_linechart.html")


# =============================================================================
# 13. LONDON BOROUGH BAR CHART (INTERACTIVE)
#
# Interactive bar chart showing average absence by London borough,
# filterable by time granularity, period, school type, and FSM status.
# Displayed inline in JupyterLab (not saved to file).
# =============================================================================

attendance_bar = pd.read_csv(derived_dir / "attendance_all.csv")

# Filter to London LA-level rows
attendance_bar = attendance_bar[
    (attendance_bar["region_name"] == "London") &
    (attendance_bar["geographic_level"] == "Local authority") &
    attendance_bar["new_la_code"].notna() &
    (attendance_bar["new_la_code"].astype(str).str.strip() != "")
].copy()

# Clean fields
for col in ["la_name", "school_type", "disadvantage_code", "time_identifier"]:
    attendance_bar[col] = attendance_bar[col].astype(str).str.strip()

attendance_bar = attendance_bar[
    attendance_bar["disadvantage_code"].isin(["FSM eligible", "FSM Not eligible"])
].copy()

METRIC = "overall_absence_perc"
attendance_bar[METRIC] = pd.to_numeric(attendance_bar[METRIC], errors="coerce")
attendance_bar = attendance_bar.dropna(subset=[METRIC])

# Build school-year label and term label for x-axis
attendance_bar["time_period_str"] = (
    attendance_bar["time_period"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)
attendance_bar = attendance_bar[
    attendance_bar["time_period_str"].str.fullmatch(r"\d{6}", na=False)
].copy()

attendance_bar["school_year_label"] = (
    attendance_bar["time_period_str"].str[:4] + "/" + attendance_bar["time_period_str"].str[4:]
)

term_order_map = {"Autumn term": 1, "Spring term": 2, "Summer term": 3}
attendance_bar["term_order"] = attendance_bar["time_identifier"].map(term_order_map)
attendance_bar = attendance_bar.dropna(subset=["term_order"])
attendance_bar["term_label"] = attendance_bar["school_year_label"] + " " + attendance_bar["time_identifier"]

# Filter option sets
time_gran_options  = ["School year", "School term"]
school_type_options = ["All", "Primary", "Secondary", "Special"]
fsm_options        = ["All", "FSM eligible", "FSM Not eligible"]
year_periods       = ["All years"]  + sorted(attendance_bar["school_year_label"].unique().tolist())
term_periods       = ["All terms"]  + sorted(attendance_bar["term_label"].unique().tolist())

def agg_borough(df):
    """Return mean absence per borough, sorted descending."""
    return (
        df.groupby("la_name", as_index=False)[METRIC]
          .mean()
          .rename(columns={METRIC: "value"})
          .sort_values("value", ascending=False)
    )

# Pre-compute all filter combinations for fast JS updates
data_map = {}
for gran in time_gran_options:
    periods = year_periods if gran == "School year" else term_periods
    for period in periods:
        for st in school_type_options:
            for fsm in fsm_options:
                df = attendance_bar.copy()
                if gran == "School year" and period != "All years":
                    df = df[df["school_year_label"] == period]
                if gran == "School term" and period != "All terms":
                    df = df[df["term_label"] == period]
                if st != "All":
                    df = df[df["school_type"] == st]
                if fsm != "All":
                    df = df[df["disadvantage_code"] == fsm]
                if len(df) == 0:
                    boroughs, values = [], []
                else:
                    agg = agg_borough(df)
                    boroughs = agg["la_name"].tolist()
                    values   = [float(x) for x in agg["value"].tolist()]
                data_map[f"{gran}|||{period}|||{st}|||{fsm}"] = {"borough": boroughs, "value": values}

# Initialise with School term / All terms / All / All
init_key  = "School term|||All terms|||All|||All"
init_data = data_map[init_key]

source_bar = ColumnDataSource(data=dict(borough=init_data["borough"], value=init_data["value"]))

p_bar = figure(
    x_range=source_bar.data["borough"],
    height=520, width=1050,
    title="London boroughs: overall absence (%)",
    x_axis_label="Borough",
    y_axis_label="Average overall absence (%)",
    tools="pan,wheel_zoom,reset",
    toolbar_location="above"
)

bars = p_bar.vbar(x="borough", top="value", width=0.9, source=source_bar)
p_bar.add_tools(HoverTool(renderers=[bars], tooltips=[("Borough", "@borough"), ("Absence (%)", "@value{0.00}")]))
p_bar.xaxis.major_label_orientation = 1.1
p_bar.y_range.start = 0

# Widgets
gran_select    = Select(title="Time granularity", value="School term", options=time_gran_options)
period_select  = Select(title="School term",      value="All terms",   options=term_periods)
school_select  = Select(title="School Type",      value="All",         options=school_type_options)
fsm_select_bar = Select(title="FSM",              value="All",         options=fsm_options)

callback_bar = CustomJS(
    args=dict(
        source=source_bar, plot=p_bar,
        gran=gran_select, period=period_select,
        school=school_select, fsm=fsm_select_bar,
        data_map=data_map,
        year_periods=year_periods, term_periods=term_periods
    ),
    code="""
    function setPeriodOptions() {
        if (gran.value === "School year") {
            period.title   = "School year";
            period.options = year_periods;
            if (!year_periods.includes(period.value)) period.value = "All years";
        } else {
            period.title   = "School term";
            period.options = term_periods;
            if (!term_periods.includes(period.value)) period.value = "All terms";
        }
    }
    function updateData() {
        const key = `${gran.value}|||${period.value}|||${school.value}|||${fsm.value}`;
        const d   = data_map[key] || {borough: [], value: []};
        source.data = { borough: d.borough, value: d.value };
        source.change.emit();
        plot.x_range.factors = d.borough;
        plot.title.text = `London boroughs: overall absence (%) — ${gran.value}: ${period.value}, School Type: ${school.value}, FSM: ${fsm.value}`;
        plot.change.emit();
    }
    setPeriodOptions();
    updateData();
    """
)

for widget in [gran_select, period_select, school_select, fsm_select_bar]:
    widget.js_on_change("value", callback_bar)

show(column(row(gran_select, period_select), row(school_select, fsm_select_bar), p_bar))